1. Загрузите данные из файла data-logistic.csv

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

data = pd.read_csv('data-logistic.csv', header=None)
X = data.iloc[:, 1:].values
y = data.iloc[:, 0].values

3. Реализуйте градиентный спуск для обычной и L2-регуляризованной
(с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения
используйте вектор (0, 0).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_descent(X, y, C=0, k=0.1, max_iter=10000, tol=1e-5, w1=0.0, w2=0.0):
    n = len(y)

    for iteration in range(max_iter):
        # Линейные комбинации: y_i * (w1*x1_i + w2*x2_i)
        margins = y * (w1 * X[:, 0] + w2 * X[:, 1])

        # Общий множитель: y_i * (1 - sigmoid(margin_i))
        common = y * (1 - sigmoid(margins))

        # Градиентный шаг
        new_w1 = w1 + k * (1/n) * np.sum(common * X[:, 0]) - k * C * w1
        new_w2 = w2 + k * (1/n) * np.sum(common * X[:, 1]) - k * C * w2

        # Проверка сходимости (евклидово расстояние между итерациями)
        dist = np.sqrt((new_w1 - w1)**2 + (new_w2 - w2)**2)
        w1, w2 = new_w1, new_w2

        if dist < tol:
            print(f"Сходимость достигнута на итерации {iteration + 1}")
            break
    else:
        print(f"Достигнут лимит итераций ({max_iter})")

    return w1, w2

def predict_proba(X, w1, w2):
    return sigmoid(w1 * X[:, 0] + w2 * X[:, 1])

4. Запустите градиентный спуск и доведите до сходимости (евклидово
расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). 

In [ ]:
w1_noreg, w2_noreg = gradient_descent(X, y, C=0, k=0.1)
proba_noreg = predict_proba(X, w1_noreg, w2_noreg)



w1_reg, w2_reg = gradient_descent(X, y, C=10, k=0.1)
proba_reg = predict_proba(X, w1_reg, w2_reg)


5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? 

In [ ]:
auc_noreg = roc_auc_score(y, proba_noreg)
auc_reg = roc_auc_score(y, proba_reg)
print(f"{auc_noreg} {auc_reg}")

6.  Попробуйте поменять длину шага. Будет ли сходиться алгоритм,
если делать более длинные шаги? Как меняется число итераций
при уменьшении длины шага?


In [ ]:
for k in [0.01, 0.1, 0.5, 1.0]:
    w1, w2 = gradient_descent(X, y, C=0, k=k)
    proba = predict_proba(X, w1, w2)
    auc = roc_auc_score(y, proba)
    print(f"k={k}: AUC-ROC={auc:.3f}")
print("Не будет, число итераций увеличивается")

7. Попробуйте менять начальное приближение. Влияет ли оно на чтонибудь?


In [ ]:
for i in np.power(10.0, np.arange(-3, 2)):
    for j in np.power(10.0, np.arange(-3, 2)):
        print(gradient_descent(X, y, np.array([i, j]), 0.1, 10))
print('Изменится кол-во итераций')